# Random Business Names Sample

This notebook selects 50,000 reproducible random business records from the train and test source files without loading the entire dataset into memory.

In [1]:
from pathlib import Path
import csv
import random
import pandas as pd

SAMPLE_SIZE = 50_000
RANDOM_SEED = 42

dataset_candidates = [
    Path.cwd() / 'student_resource' / 'dataset',
    Path.cwd().parent / 'student_resource' / 'dataset',
    Path.cwd().parent.parent / 'student_resource' / 'dataset',
]
DATASET_DIR = next((path for path in dataset_candidates if path.exists()), None)
if DATASET_DIR is None:
    raise FileNotFoundError('Could not find student_resource/dataset')

source_files = sorted(list((DATASET_DIR / 'train').glob('train_source*.tsv')) +
                     list((DATASET_DIR / 'test').glob('test_source*.tsv')))
source_files

[WindowsPath('d:/deep_learning/student_resource/dataset/test/test_source1.tsv'),
 WindowsPath('d:/deep_learning/student_resource/dataset/test/test_source2.tsv'),
 WindowsPath('d:/deep_learning/student_resource/dataset/test/test_source3.tsv'),
 WindowsPath('d:/deep_learning/student_resource/dataset/train/train_source1.tsv'),
 WindowsPath('d:/deep_learning/student_resource/dataset/train/train_source2.tsv'),
 WindowsPath('d:/deep_learning/student_resource/dataset/train/train_source3.tsv')]

In [2]:
rng = random.Random(RANDOM_SEED)
reservoir = []
rows_seen = 0

for path in source_files:
    source = path.stem
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle, delimiter='\t')
        for row in reader:
            record = {
                'source': source,
                'entity_id': row['entity_id'],
                'business_name': row['business_name'],
                'business_address': row['business_address'],
                'country': row['country'],
            }
            rows_seen += 1
            if len(reservoir) < SAMPLE_SIZE:
                reservoir.append(record)
            else:
                replacement = rng.randrange(rows_seen)
                if replacement < SAMPLE_SIZE:
                    reservoir[replacement] = record

sample = pd.DataFrame(reservoir)
print(f'Rows scanned: {rows_seen:,}')
print(f'Random records selected: {len(sample):,}')
sample.head(20)

Rows scanned: 24,229,173
Random records selected: 50,000


,source,entity_id,business_name,business_address,country
0,train_source2,S2-582986458,Mcnair Prairie Railway LLC,"1322 PINE AVE, ARLINTON HEIGHTS, IL",US
1,test_source2,S2-399742206,Sàrl Tourcoing Centre,"Hauts-de-France, 64 RUE DE CAMBRAI, Tourcoing",France
2,train_source2,S2-261452351,Roy Buildcon,"HN 7-184 PLOT NO. T 99, KAGAL FIVE STAR MIDC K...",India
3,test_source3,S3-587568684,Bajrang Digital Limited,"Nr. Ladbi Nala, Banaskantha, Palanpur, GJ",India
4,train_source2,S2-931129485,Shri Jsk Sa1es Private Ltd,"SHED NO. C-D/19 INDUSTRIAL ESTATE, RAICHUR, Ka...",India
5,train_source1,S1-133984395,"Ahrendt Permian, LLC","86 Mallow Lane, Roanoke Rapids, NC",US
6,train_source1,S1-881583817,West Bengal Service Private Limited,"Ukilpara Behind Hospital Baruipur, Parganas So...",India
7,test_source2,S2-878226314,M/s application resolution private limited,"NO 23-2-4 , BLOCK NO 23, VANAPALLI VARI STREET...",India
8,train_source3,S3-892926089,Interstate Foundation North LLC Service,"6707 Guthrie Avenue, Cleveland, Ohio",US
9,train_source3,S3-132266775,ZN Átcie Private Limited,"AP, 09-6A, Bhavanipuram Colony Lalupuram, Gunt...",India


In [3]:
print('Records by source:')
display(sample['source'].value_counts().rename_axis('source').to_frame('records'))
print('Records by country:')
display(sample['country'].value_counts().rename_axis('country').to_frame('records'))
print('Missing values:')
display(sample.isna().sum().rename('missing').to_frame())

Records by source:


,records
source,
train_source3,10938
test_source3,10494
train_source2,10379
test_source2,10091
train_source1,4526
test_source1,3572


Records by country:


,records
country,
US,24886
India,21609
France,3505


Missing values:


,missing
source,0
entity_id,0
business_name,0
business_address,0
country,0


In [4]:
output_dir = Path.cwd() / 'team-AiseKaise-amlc26' / 'output'
if not output_dir.parent.exists():
    output_dir = DATASET_DIR.parent.parent / 'team-AiseKaise-amlc26' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'random_business_names_50000.csv'
sample.to_csv(output_path, index=False, encoding='utf-8')
print(f'Saved: {output_path.resolve()}')

Saved: D:\deep_learning\team-AiseKaise-amlc26\output\random_business_names_50000.csv
